In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import os, time

In [2]:
#-----------------------------------------------------------------------------#
#    IMAGE PREPROCESSING                                                      #
#-----------------------------------------------------------------------------#
## DIRECTORIES with datasets
img_path = 'dataset'

# Dataset with Transformation (using only 100 images)
full_dataset = datasets.ImageFolder(
    img_path,
    transforms.Compose([
        transforms.Resize((128, 256)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.0, 0.0, 0.0], 
            std=[1.0, 1.0, 1.0])
    ])
)
dataset = torch.utils.data.Subset(full_dataset, list(range(1000)))

print(f"Dataset size: {len(dataset)}")

Dataset size: 1000


In [3]:
#-----------------------------------------------------------------------------#
#    DATA LOADER                                                              #
#-----------------------------------------------------------------------------#
dataloader = DataLoader(dataset , batch_size=1, shuffle=False, num_workers=1)

# print(dataset)
# print(dataloader)
print("Min/Max:", dataset[0][0].min(), dataset[0][0].max())

Min/Max: tensor(0.) tensor(1.)


# Test CPU

In [4]:
class vaemodel1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.mu = nn.Linear(256, 6)  # Reduced latent space size
        self.std = nn.Linear(256, 6)  # Reduced latent space size

    def forward(self, x):
        a = self.encoder(x).permute(0,2,3,1)
        a = torch.flatten(a, start_dim=1)
        mu = self.mu(a)
        lvar = self.std(a)
        out = torch.cat((mu, lvar), dim=1)
        return out

model = vaemodel1()

In [5]:
model.load_state_dict(torch.load("pre_trained_w_encoder.pt", weights_only=True))

<All keys matched successfully>

In [6]:
model.eval()
with torch.no_grad():
    inference_time = 0
    cpu_z = []
    for image, _ in dataloader:
        # Run inference for each image individually
        start_time = time.time()
        pred = model(image)
        inference_time += time.time() - start_time
        cpu_z.append(pred)

    # Convert list to numpy array
    cpu_z = np.array(cpu_z)

    # calculate average inference time and FPS
    num_images = len(cpu_z)
    avg_inference_time = inference_time / num_images
    fps = num_images / inference_time

    print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
    print("Average inference time per image: {:.6f} seconds".format(avg_inference_time))
    print("FPS: {:.2f}".format(fps))
    cpu_inference_time = inference_time

The total inference time was 39.659269 seconds for 1000.
Average inference time per image: 0.039659 seconds
FPS: 25.21


# Test DPU

In [7]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("../vitisai_bitstream/dpu.bit")

In [8]:
overlay.load_model("zcu104_vaemodel1.xmodel")

In [9]:
dpu = overlay.runner

inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

shapeIn = tuple(inputTensors[0].dims)
shapeOut = tuple(outputTensors[0].dims)
outputSize = int(outputTensors[0].get_data_size() / shapeIn[0])

output_data = [np.empty(shapeOut, dtype=np.float32, order="C")]
input_data = [np.empty(shapeIn, dtype=np.float32, order="C")]
data = input_data[0]

In [10]:
inference_time = 0
vitisAI_z = []
for image, _ in dataloader:
    data = image.permute(0,2,3,1)
    start_time = time.time()
    job_id = dpu.execute_async(input_data, output_data)
    dpu.wait(job_id)
    inference_time += time.time() - start_time
    #o_data = output_data[0].reshape(2, 6)
    #std = np.exp(o_data[1]*0.5)
    #eps = np.random.randn(*std.shape)
    #vitisAI_z.append(o_data[0]+eps*std)
    vitisAI_z.append(output_data[0])

# Convert list to numpy array
vitisAI_z = np.array(vitisAI_z)

num_images = len(vitisAI_z)
avg_infer_time = inference_time/num_images
fps = num_images/inference_time
print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
print("Average inference time: {} s".format(avg_infer_time))
print("Performance: {} FPS".format(fps))
print(f"Speedup: {cpu_inference_time/inference_time:.2f}")

The total inference time was 1.648410 seconds for 1000.
Average inference time: 0.0016484103202819825 s
Performance: 606.6450735572541 FPS
Speedup: 24.06


In [11]:
# Calculate MSE
mse = np.mean((cpu_z - vitisAI_z) ** 2)
print(f"Mean Squared Error between CPU and VitisAI outputs: {mse:.6f}")
print(cpu_z)
print(vitisAI_z)

Mean Squared Error between CPU and VitisAI outputs: 2.818901
[[[-2.1714897   0.29388946  0.15299428 ... -6.055775   -5.117832
   -5.300388  ]]

 [[-2.2223816   0.30713335  0.14875156 ... -6.1507416  -5.156154
   -5.2966356 ]]

 [[-2.2640526   0.32817918  0.05832664 ... -6.3531566  -5.24033
   -5.333772  ]]

 ...

 [[ 1.0842187   0.12422871  1.5751365  ... -5.5208993  -4.9587173
   -5.1042423 ]]

 [[ 1.0958327  -0.06794329  1.1317456  ... -5.481964   -4.9567523
   -5.406106  ]]

 [[ 0.9963672  -0.27504882  1.0347071  ... -5.495946   -5.3573976
   -5.812979  ]]]
[[[-1.1875  2.0625  0.75   ... -7.625  -5.375  -4.8125]]

 [[-1.1875  2.0625  0.75   ... -7.625  -5.375  -4.8125]]

 [[-1.1875  2.0625  0.75   ... -7.625  -5.375  -4.8125]]

 ...

 [[-1.1875  2.0625  0.75   ... -7.625  -5.375  -4.8125]]

 [[-1.1875  2.0625  0.75   ... -7.625  -5.375  -4.8125]]

 [[-1.1875  2.0625  0.75   ... -7.625  -5.375  -4.8125]]]
